# Diabetes screening model review\n\nThis notebook is a reproducible training/evaluation template. It does not fabricate data or metrics.\n

In [ ]:
from __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\nimport pandas as pd\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (\n    accuracy_score,\n    classification_report,\n    confusion_matrix,\n    f1_score,\n    recall_score,\n    roc_auc_score,\n)\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\n

In [ ]:
FEATURES = [\n    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',\n    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age',\n]\nTARGET = 'Outcome'\nRANDOM_STATE = 42\nDATA_PATH = Path(os.environ.get('DIABETES_DATA_PATH', 'data/diabetes.csv')).expanduser()\nif not DATA_PATH.is_absolute():\n    DATA_PATH = Path.cwd() / DATA_PATH\nif not DATA_PATH.is_file():\n    raise FileNotFoundError(\n        f'Dataset not found at {DATA_PATH}. Set DIABETES_DATA_PATH to the CSV path.'\n    )\ndf = pd.read_csv(DATA_PATH)\nmissing = set(FEATURES + [TARGET]) - set(df.columns)\nif missing:\n    raise ValueError(f'Dataset is missing required columns: {sorted(missing)}')\ndf = df[FEATURES + [TARGET]].drop_duplicates().copy()\n

In [ ]:
X = df[FEATURES]\ny = df[TARGET]\nif y.isna().any() or y.nunique() != 2:\n    raise ValueError('Outcome must be non-null and binary.')\nX_train, X_test, y_train, y_test = train_test_split(\n    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE\n)\n

In [ ]:
# Fit all learned preprocessing only on the training partition.\nmodel = Pipeline(steps=[\n    ('imputer', SimpleImputer(strategy='median')),\n    ('scaler', StandardScaler()),\n    ('classifier', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),\n])\nmodel.fit(X_train, y_train)\npredicted = model.predict(X_test)\nprobability = model.predict_proba(X_test)[:, 1]\n

In [ ]:
metrics = {\n    'accuracy': accuracy_score(y_test, predicted),\n    'recall': recall_score(y_test, predicted),\n    'f1': f1_score(y_test, predicted),\n    'roc_auc': roc_auc_score(y_test, probability),\n]\nprint(metrics)\nprint(confusion_matrix(y_test, predicted))\nprint(classification_report(y_test, predicted, zero_division=0))\n

## Reproducibility notes\n\nThe test split is stratified and seeded. Metrics are calculated with `y_test` as the ground truth. No oversampling is applied; if it is added later, it must be fitted inside the training pipeline only. The checked-in `model.pkl` is an existing inference artifact and is not overwritten by this notebook.\n